# Task 2A — Synthetic Dataset Engineering: Skin Symptom-Intake Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<GITHUB_USER>/CDAZZDEV-MLE-Nipuna-Tennakoon/blob/main/task2_genai/notebooks/01_data_generation.ipynb)

This notebook builds the fine-tuning dataset for **Task 2 — Generative AI**.

| Step | What happens |
|---|---|
| 1 | Environment, secrets, and paths |
| 2 | Use-case definition (problem statement) |
| 3 | Teacher-model pool (free tier: Groq + OpenRouter) |
| 4 | Generation with a stratified *(stage × concern)* job plan |
| 5 | Quality gate + exact and semantic de-duplication |
| 6 | Diversity analysis (prompt-length distribution, topic & keyword frequency) |
| 7 | Stratified 80 / 10 / 10 split, saved as chat-format JSONL |
| 8 | Verification: contract audit, cross-split leakage, chat-template render |
| A | Appendix: full teacher prompt |

> **Teacher ≠ student.** Data is generated by **Llama-3.3-70B / GPT-OSS-120B** (teachers).
> The student fine-tuned in `02_finetuning.ipynb` is **Qwen2.5-1.5B-Instruct** — a different model family.

## 1) Environment, Secrets and Paths

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Create a data generation notebook adapted from my llm-fine-tuning-medical-chat project', Date: 2026-10-10
import sys, os, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
GITHUB_REPO_URL = "https://github.com/<GITHUB_USER>/CDAZZDEV-MLE-Nipuna-Tennakoon.git"  # set before running on Colab

if IN_COLAB:
    repo_dir = Path("/content/CDAZZDEV-MLE-Nipuna-Tennakoon")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir / "task2_genai" / "notebooks")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "openai>=1.50", "python-dotenv", "transformers>=4.56", "matplotlib", "pandas"],
        check=True,
    )

In [ ]:
import hashlib
import json
import random
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoModel, AutoTokenizer


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until the folder holding pyproject.toml is found."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Could not locate repo root (pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
TASK_DIR = REPO_ROOT / "task2_genai"
DATA_DIR = TASK_DIR / "data"
PROMPT_PATH = TASK_DIR / "prompts" / "teacher_skin_intake.md"
REPORTS_DIR = TASK_DIR / "reports"
for d in (DATA_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO_ROOT / ".env")


def get_secret(name: str) -> str | None:
    """Read a secret from Colab Secrets first, then from the environment / .env file.
    Keys are never hard-coded in the notebook (assessment rule 2.3)."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return os.getenv(name) or None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"Prompt    : {PROMPT_PATH}")

## 2) Use-Case Definition (Problem Statement)

**Domain:** medical triage — *skin symptom intake* before a dermatology / GP consultation.

| | |
|---|---|
| **Input** | A fixed system prompt + a short patient ↔ agent conversation (1–4 exchanges). The patient describes a skin, scalp, nail, hair, or wound concern in informal language. |
| **Output** | The **next assistant turn**: a short (≤ 45 words), calm reply that either (a) asks the single most useful *missing* symptom question, or (b) when enough is known, recaps the symptoms in one sentence and gives one safe next step. |
| **Correct response** | Acknowledges the patient; asks about an **unfilled** symptom slot (location, appearance, duration, sensation, drainage, systemic signs, triggers, history, care so far); stays short; no diagnosis; no medication; escalates red flags (e.g. facial swelling + breathing trouble) to emergency care. |
| **Incorrect response** | Names a condition (*"this is eczema"*), suggests any cream / drug / home remedy, repeats a question already answered, asks several unrelated questions at once, is long or preachy, misses a red flag, or follows an off-topic / instruction-changing request. |

Why this is non-trivial: the model must track **what is already known** across turns, pick
the next best question, decide **when to stop**, and hold strict safety boundaries — general
models tend to diagnose and recommend treatments instead.

Two conversation states are generated:

| stage | last assistant turn | share |
|---|---|---|
| `intermediate` | ends with a symptom question — intake in progress | 55 % |
| `final` | one-sentence recap + one safe next step (no question) | 45 % |

## 3) Teacher-Model Pool and Generation Config

In [ ]:
# Single canonical system prompt — identical at train and inference time.
SYSTEM_PROMPT = (
    "You are a calm and polite skin symptom intake assistant. Ask short, focused questions "
    "to gather the patient's skin symptoms one at a time. Do not diagnose and do not "
    "recommend medications."
)

TOTAL_SAMPLES = 300          # target AFTER de-duplication (assessment minimum is 100)
GENERATION_HEADROOM = 1.3    # over-generate; semantic dedup trims the surplus
SAMPLES_PER_API_CALL = 8     # small batches keep free-tier output-token limits safe and quality high
MAX_RETRIES = 4
MAX_CALLS = 120              # hard budget on API calls (free-tier friendly backstop)
TEMPERATURE = 0.9            # high, for scenario diversity
MAX_OUTPUT_TOKENS = 4096

STAGE_MIX = {"intermediate": 0.55, "final": 0.45}
EXCHANGE_RANGE = {"intermediate": (1, 3), "final": (2, 4)}   # user/assistant pairs per stage
MAX_ASSISTANT_WORDS = 60
MAX_USER_WORDS = 70

# Concern taxonomy. Each API call is pinned to ONE concern so coverage is guaranteed by
# design rather than hoped for — this is what drives the topic-diversity metrics in section 6.
CONCERNS: dict[str, str] = {
    "rash_itch_dryness": "rash, irritation, itching or dry skin",
    "acne_breakouts": "acne and facial breakouts",
    "eczema_sensitive_flare": "eczema-like or sensitive-skin flare-ups",
    "ring_fungal_patch": "ring-shaped or fungal-looking patches",
    "hives_allergic": "hives, swelling or allergic-looking reactions",
    "wound_burn_bite": "wounds, cuts, burns or insect bites with possible infection",
    "nail_changes": "nail changes (colour, thickening, separation)",
    "hair_scalp": "hair loss, scalp flaking or itchy scalp",
    "changing_mole": "moles or spots that are changing",
    "sun_damage": "sunburn and sun damage",
    "pigmentation": "dark or light patches and pigmentation changes",
    "red_flag_emergency": "red-flag emergencies with skin involvement (escalate to emergency care)",
    "off_topic_redirect": "patients who go off-topic or try to change the agent's instructions (politely redirect)",
}

PERSONAS = [
    "a teenager", "a young adult", "a working adult", "an older adult (70+)",
    "a parent asking about a toddler", "a parent asking about a school-age child",
    "a pregnant adult", "an athlete", "an outdoor worker", "a person with a busy night-shift job",
]

# Free-tier, OpenAI-compatible teachers. Any provider whose key is missing is skipped.
# `min_interval_s` throttles calls per model to stay under free-tier requests-per-minute limits.
TEACHER_POOL: list[dict[str, Any]] = [
    {"provider": "groq", "model": "llama-3.3-70b-versatile",
     "base_url": "https://api.groq.com/openai/v1", "key_env": "GROQ_API_KEY", "min_interval_s": 4},
    {"provider": "groq", "model": "openai/gpt-oss-120b",
     "base_url": "https://api.groq.com/openai/v1", "key_env": "GROQ_API_KEY", "min_interval_s": 4},
    {"provider": "openrouter", "model": "meta-llama/llama-3.3-70b-instruct:free",
     "base_url": "https://openrouter.ai/api/v1", "key_env": "OPENROUTER_API_KEY", "min_interval_s": 8},
]

active_teachers: list[dict[str, Any]] = []
for spec in TEACHER_POOL:
    key = get_secret(spec["key_env"])
    if not key:
        print(f"  [skip] {spec['model']}: {spec['key_env']} not set")
        continue
    spec = {**spec, "client": OpenAI(api_key=key, base_url=spec["base_url"]), "last_call": 0.0}
    active_teachers.append(spec)

if not active_teachers:
    raise RuntimeError("No teacher model available — set GROQ_API_KEY and/or OPENROUTER_API_KEY")

print(f"Active teachers: {[t['model'] for t in active_teachers]}")
print(f"Target after dedup: {TOTAL_SAMPLES}  |  per call: {SAMPLES_PER_API_CALL}")

## 4) Helpers — Prompt Building, Parsing, Quality Gate

`normalize_record` is the **quality gate**. A record is kept only if it satisfies every rule
the student must learn:

- strict `system → user → assistant → …` alternation, ending on `assistant`
- turn lengths within limits; exchange count within `EXCHANGE_RANGE` for its stage
- `intermediate` ends with a question, `final` does not
- no banned meta language (*"as an AI"*, *"policy"* …) and no medication / diagnosis wording

In [ ]:
PROMPT_TEMPLATE = PROMPT_PATH.read_text(encoding="utf-8")

BANNED_ASSISTANT_PATTERNS = [
    r"\bas an ai\b", r"\bai (model|assistant|language)\b", r"\blanguage model\b",
    r"\bpolicy\b", r"\bguidelines\b", r"\bi'?m not a (doctor|medical)", r"\bdisclaimer\b",
    r"\bi cannot\b",
    # treatment / diagnosis leakage
    r"\b(cream|ointment|antibiotic|steroid|hydrocortisone|antihistamine|ibuprofen|paracetamol|dose|mg)\b",
    r"\byou (have|probably have|might have)\b", r"\bthis (is|looks like|sounds like) (an? )?(eczema|psoriasis|ringworm|fungal|infection|melanoma|acne)\b",
]
BANNED_RE = re.compile("|".join(BANNED_ASSISTANT_PATTERNS), re.IGNORECASE)


def build_prompt(num_samples: int, stage: str, concern_tag: str) -> str:
    personas = ", ".join(random.sample(PERSONAS, k=4))
    return (
        PROMPT_TEMPLATE
        .replace("{{NUM_SAMPLES}}", str(num_samples))
        .replace("{{STAGE}}", stage)
        .replace("{{CONCERN}}", CONCERNS[concern_tag])
        .replace("{{CONCERN_TAG}}", concern_tag)
        .replace("{{PERSONAS}}", personas)
        .replace("{{SYSTEM_PROMPT}}", SYSTEM_PROMPT)
    )


def parse_jsonl_objects(raw_text: str) -> list[dict[str, Any]]:
    """Parse one JSON object per line, tolerating markdown fences and trailing commas."""
    records = []
    for line in raw_text.splitlines():
        line = line.strip().rstrip(",")
        if not line or line.startswith("```"):
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            records.append(obj)
        elif isinstance(obj, list):
            records.extend(o for o in obj if isinstance(o, dict))
    return records


def normalize_record(obj: dict[str, Any], teacher: str, stage: str, concern: str) -> dict[str, Any] | None:
    """Validate + normalize one generated record. Returns None to reject it."""
    messages = obj.get("messages")
    if not isinstance(messages, list):
        return None
    turns = [m for m in messages if isinstance(m, dict) and m.get("role") != "system"]
    if len(turns) < 2 or len(turns) % 2:
        return None
    lo, hi = EXCHANGE_RANGE[stage]
    if not lo <= len(turns) // 2 <= hi:
        return None

    cleaned = [{"role": "system", "content": SYSTEM_PROMPT}]
    for i, msg in enumerate(turns):
        role = "user" if i % 2 == 0 else "assistant"
        content = msg.get("content")
        if msg.get("role") != role or not isinstance(content, str) or not content.strip():
            return None
        content = re.sub(r"\s+", " ", content.strip())
        limit = MAX_USER_WORDS if role == "user" else MAX_ASSISTANT_WORDS
        if len(content.split()) > limit:
            return None
        if role == "assistant" and BANNED_RE.search(content):
            return None
        cleaned.append({"role": role, "content": content})

    ends_with_q = cleaned[-1]["content"].rstrip().endswith("?")
    if (stage == "intermediate") != ends_with_q:
        return None

    return {"id": "", "stage": stage, "concern": concern, "teacher": teacher, "messages": cleaned}


def record_fingerprint(rec: dict[str, Any]) -> str:
    """md5 over dialogue turns (system prompt excluded) for exact-duplicate removal."""
    text = " ".join(m["content"] for m in rec["messages"] if m["role"] != "system").lower()
    return hashlib.md5(re.sub(r"\s+", " ", text).encode()).hexdigest()

### Teacher call with retry, back-off and free-tier throttling

Errors are **returned, not raised**: one teacher hitting its quota must not kill a run the
other teachers can still finish. Fatal errors (bad key, unknown model) retire the teacher;
rate-limit errors (HTTP 429) back off exponentially.

In [ ]:
FATAL_MARKERS = ("invalid_api_key", "authentication", "model_not_found", "does not exist",
                 "decommissioned", "not a valid model", "401", "403", "404")
RATE_LIMIT_MARKERS = ("429", "rate limit", "rate_limit", "too many requests")


def _is_fatal(err: Exception) -> bool:
    text = str(err).lower()
    return any(m in text for m in FATAL_MARKERS)


def _is_rate_limited(err: Exception) -> bool:
    text = str(err).lower()
    return any(m in text for m in RATE_LIMIT_MARKERS)


def call_teacher(spec: dict[str, Any], prompt: str) -> str:
    wait = spec["min_interval_s"] - (time.time() - spec["last_call"])
    if wait > 0:
        time.sleep(wait)
    spec["last_call"] = time.time()
    resp = spec["client"].chat.completions.create(
        model=spec["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_OUTPUT_TOKENS,
    )
    return (resp.choices[0].message.content or "").strip()


def generate_batch(spec, num_samples, stage, concern):
    """One teacher call -> (accepted_records, n_parsed, error)."""
    prompt = build_prompt(num_samples, stage, concern)
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            parsed = parse_jsonl_objects(call_teacher(spec, prompt))
            recs = [r for r in (normalize_record(o, spec["model"], stage, concern) for o in parsed) if r]
            return recs, len(parsed), None
        except Exception as e:  # noqa: BLE001 — surfaced to the caller as a value
            last_err = e
            if _is_fatal(e):
                break
            time.sleep((20 if _is_rate_limited(e) else 2) * (2 ** attempt))
    return [], 0, last_err

## 5) Generate with a Stratified (stage × concern) Job Plan

Every *(stage, concern)* cell gets a quota. Each round, every active teacher takes the cell
that is **furthest behind its quota**, so stage mix, topic coverage and teacher mix stay
balanced. Exact duplicates are dropped as records arrive.

In [ ]:
GENERATION_TARGET = int(TOTAL_SAMPLES * GENERATION_HEADROOM)
cells = [(s, c) for s in STAGE_MIX for c in CONCERNS]
cell_quota = {(s, c): GENERATION_TARGET * STAGE_MIX[s] / len(CONCERNS) for s, c in cells}
cell_count = Counter()

all_records: list[dict[str, Any]] = []
seen: set[str] = set()
generation_log: list[dict[str, Any]] = []
retired: dict[str, str] = {}
calls_made = 0


def next_cell(taken: set) -> tuple[str, str]:
    """Cell with the lowest fill ratio that is not already assigned this round."""
    free = [c for c in cells if c not in taken] or cells
    return min(free, key=lambda c: cell_count[c] / cell_quota[c])


t0 = time.time()
while len(all_records) < GENERATION_TARGET and active_teachers and calls_made < MAX_CALLS:
    taken: set = set()
    jobs = []
    for spec in active_teachers:
        cell = next_cell(taken)
        taken.add(cell)
        jobs.append((spec, cell))

    with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
        futures = {pool.submit(generate_batch, spec, SAMPLES_PER_API_CALL, *cell): (spec, cell)
                   for spec, cell in jobs}
        for fut in as_completed(futures):
            spec, (stage, concern) = futures[fut]
            recs, n_parsed, err = fut.result()
            calls_made += 1
            new = 0
            for r in recs:
                fp = record_fingerprint(r)
                if fp not in seen:
                    seen.add(fp)
                    all_records.append(r)
                    cell_count[(stage, concern)] += 1
                    new += 1
            generation_log.append({"teacher": spec["model"], "stage": stage, "concern": concern,
                                   "parsed": n_parsed, "accepted": len(recs), "new": new,
                                   "error": str(err)[:120] if err else None})
            if err is not None and _is_fatal(err):
                retired[spec["model"]] = str(err)[:160]
            print(f"{spec['model'][:34]:<34} {stage:<12} {concern:<24} parsed={n_parsed:>2} "
                  f"accepted={len(recs):>2} new={new:>2}" + (f"  ERR {str(err)[:60]}" if err else ""))

    active_teachers = [t for t in active_teachers if t["model"] not in retired]
    print(f"--> total={len(all_records)}/{GENERATION_TARGET}  calls={calls_made}  "
          f"elapsed={time.time() - t0:.0f}s\n")

for i, r in enumerate(all_records, 1):
    r["id"] = f"skin_intake_{i:05d}"

if retired:
    print("Retired teachers:", retired)
print(f"Generated {len(all_records)} unique records with {calls_made} API calls")

In [ ]:
# Generation health: acceptance rate per teacher shows how well each follows the format contract.
gen_df = pd.DataFrame(generation_log)
summary = gen_df.groupby("teacher")[["parsed", "accepted", "new"]].sum()
summary["acceptance_rate"] = (summary["accepted"] / summary["parsed"].clip(lower=1)).round(3)
display(summary)

# Save the raw (pre-semantic-dedup) pool for reproducibility.
with (DATA_DIR / "raw_generated.jsonl").open("w", encoding="utf-8") as f:
    for r in all_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

## 6) Semantic Near-Duplicate Removal

Exact duplicates are gone; *semantic* near-duplicates remain (e.g. many "itchy red patch on
the forearm" variants). Dedup runs **before** the split so a near-twin cannot leak across
train / test.

Method: embed the dialogue turns with `NeuML/pubmedbert-base-embeddings` (biomedical
sentence embeddings), then greedily keep a record only if its cosine similarity to every
kept record is below `NEAR_DUP_THRESHOLD`.

In [ ]:
EMBED_MODEL = "NeuML/pubmedbert-base-embeddings"
EMBED_BATCH = 32
NEAR_DUP_THRESHOLD = 0.95

_device = "cuda" if torch.cuda.is_available() else "cpu"
embed_tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(EMBED_MODEL).to(_device).eval()


def record_to_text(rec: dict[str, Any]) -> str:
    return "\n".join(f"{m['role']}: {m['content']}" for m in rec["messages"] if m["role"] != "system")


@torch.no_grad()
def embed_texts(texts: list[str]) -> np.ndarray:
    """Masked mean-pooled, L2-normalised embeddings."""
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        enc = embed_tok(texts[i:i + EMBED_BATCH], padding=True, truncation=True,
                        max_length=512, return_tensors="pt").to(_device)
        hidden = embed_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
        out.append(torch.nn.functional.normalize(pooled.float(), dim=1).cpu().numpy())
    return np.concatenate(out)


pool_emb = embed_texts([record_to_text(r) for r in all_records])

kept_idx, dropped = [], []
kept_mat = np.zeros((0, pool_emb.shape[1]), dtype=np.float32)
for i, v in enumerate(pool_emb):
    if len(kept_idx):
        sims = kept_mat @ v
        j = int(sims.argmax())
        if sims[j] >= NEAR_DUP_THRESHOLD:
            dropped.append((i, kept_idx[j], float(sims[j])))
            continue
    kept_idx.append(i)
    kept_mat = np.vstack([kept_mat, v[None]])

deduped = [all_records[i] for i in kept_idx]
deduped_emb = kept_mat
print(f"Near-duplicates removed (cos >= {NEAR_DUP_THRESHOLD}): {len(dropped)}")
print(f"Remaining: {len(deduped)}")
for i, j, s in sorted(dropped, key=lambda d: -d[2])[:3]:
    print(f"\n cos={s:.3f}\n  dropped: {record_to_text(all_records[i])[:120]}…\n  kept   : {record_to_text(all_records[j])[:120]}…")

In [ ]:
# Stratified trim to TOTAL_SAMPLES: keep the stage mix, then round-robin across concerns so
# no topic is over-represented. If we are short, keep everything and warn.
if len(deduped) > TOTAL_SAMPLES:
    rng = random.Random(SEED)
    final_idx = []
    for stage, w in STAGE_MIX.items():
        quota = round(TOTAL_SAMPLES * w)
        buckets = {}
        for k, r in enumerate(deduped):
            if r["stage"] == stage:
                buckets.setdefault(r["concern"], []).append(k)
        queues = [rng.sample(b, len(b)) for _, b in sorted(buckets.items())]
        picked = []
        while len(picked) < quota and any(queues):
            for q in queues:
                if q and len(picked) < quota:
                    picked.append(q.pop())
        final_idx += picked
    final_idx = sorted(final_idx)[:TOTAL_SAMPLES]
    dataset = [deduped[k] for k in final_idx]
    dataset_emb = deduped_emb[final_idx]
else:
    dataset, dataset_emb = deduped, deduped_emb
    if len(dataset) < TOTAL_SAMPLES:
        print(f"⚠ only {len(dataset)} records (< {TOTAL_SAMPLES}). Raise MAX_CALLS / headroom and re-run section 5.")

assert len(dataset) >= 100, "Assessment requires at least 100 training examples"
print(f"Final dataset: {len(dataset)} records")
print("Stage  :", dict(Counter(r["stage"] for r in dataset)))
print("Teacher:", dict(Counter(r["teacher"] for r in dataset)))

## 7) Diversity Analysis

The assessment requires proof that the data is **not** minor variations of one scenario.
We report:

1. **Prompt-length distribution** — words in the first patient message and in the full prompt
   (everything before the target assistant turn), plus number of exchanges.
2. **Topic frequency** — records per concern category and per stage.
3. **Keyword frequency** — top content words in patient turns.
4. **Lexical diversity** — distinct-1 / distinct-2 (unique uni/bi-grams ÷ total) and opener variety.
5. **Semantic diversity** — mean nearest-neighbour cosine similarity (lower = more diverse).

In [ ]:
STOPWORDS = set("""a an the and or but if of to in on at for with from by is are was were be been
being it its it's this that these those i me my we our you your he she they them his her their
have has had do does did not no yes so very just about like also can could would should will
there here what when where which who how than then too up down out over again some any all
im i'm ive i've dont don't its it's get got really bit little now one two since week weeks day
days ago am thanks thank please okay ok hi hello""".split())


def first_user(rec):  return next(m["content"] for m in rec["messages"] if m["role"] == "user")
def prompt_text(rec): return " ".join(m["content"] for m in rec["messages"][:-1])
def tokens(text):     return re.findall(r"[a-z']+", text.lower())

stats_df = pd.DataFrame({
    "stage": [r["stage"] for r in dataset],
    "concern": [r["concern"] for r in dataset],
    "teacher": [r["teacher"] for r in dataset],
    "first_user_words": [len(first_user(r).split()) for r in dataset],
    "prompt_words": [len(prompt_text(r).split()) for r in dataset],
    "target_words": [len(r["messages"][-1]["content"].split()) for r in dataset],
    "exchanges": [(len(r["messages"]) - 1) // 2 for r in dataset],
})
display(stats_df.describe().round(1))

user_tokens = [t for r in dataset for m in r["messages"] if m["role"] == "user"
               for t in tokens(m["content"]) if t not in STOPWORDS and len(t) > 2]
keyword_freq = Counter(user_tokens).most_common(25)


def distinct_n(texts, n):
    grams = [tuple(tk[i:i + n]) for t in texts for tk in [tokens(t)] for i in range(len(tk) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)


assistant_texts = [m["content"] for r in dataset for m in r["messages"] if m["role"] == "assistant"]
user_texts = [m["content"] for r in dataset for m in r["messages"] if m["role"] == "user"]
openers = [" ".join(tokens(t)[:3]) for t in assistant_texts]

sim = dataset_emb @ dataset_emb.T
np.fill_diagonal(sim, -1)
nn_sim = sim.max(axis=1)

diversity = {
    "n_records": len(dataset),
    "n_concerns_covered": int(stats_df["concern"].nunique()),
    "max_concern_share": round(float(stats_df["concern"].value_counts(normalize=True).max()), 3),
    "user_distinct_1": round(distinct_n(user_texts, 1), 3),
    "user_distinct_2": round(distinct_n(user_texts, 2), 3),
    "assistant_distinct_2": round(distinct_n(assistant_texts, 2), 3),
    "assistant_unique_opener_ratio": round(len(set(openers)) / len(openers), 3),
    "mean_nn_cosine": round(float(nn_sim.mean()), 3),
    "max_nn_cosine": round(float(nn_sim.max()), 3),
}
print(json.dumps(diversity, indent=2))
print("\nTop keywords:", keyword_freq[:15])

In [ ]:
# Validated categorical palette, consistent across all charts in this repo.
C_BLUE, C_ORANGE, C_GREEN, C_PURPLE, INK, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7", "#52514e", "#e1e0d9"
STAGE_COLORS = {"intermediate": C_BLUE, "final": C_ORANGE}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Dataset diversity — skin symptom-intake conversations", fontsize=13)

ax = axes[0, 0]
for st, c in STAGE_COLORS.items():
    ax.hist(stats_df.loc[stats_df.stage == st, "prompt_words"], bins=25, alpha=0.6, color=c, label=st)
ax.set_title("Prompt length (words before target turn)"); ax.set_xlabel("words"); ax.set_ylabel("records"); ax.legend(frameon=False)

ax = axes[0, 1]
ax.hist(stats_df["first_user_words"], bins=25, color=C_GREEN, alpha=0.8)
ax.set_title("First patient message length"); ax.set_xlabel("words"); ax.set_ylabel("records")

ax = axes[1, 0]
ct = pd.crosstab(stats_df.concern, stats_df.stage).loc[lambda d: d.sum(1).sort_values().index]
ct.plot.barh(stacked=True, ax=ax, color=[STAGE_COLORS[c] for c in ct.columns], width=0.75)
ax.set_title("Topic frequency (concern × stage)"); ax.set_xlabel("records"); ax.set_ylabel(""); ax.legend(frameon=False)

ax = axes[1, 1]
words, counts = zip(*keyword_freq[:20])
ax.barh(list(words)[::-1], list(counts)[::-1], color=C_PURPLE)
ax.set_title("Top-20 keywords in patient turns"); ax.set_xlabel("count")

for a in axes.ravel():
    a.grid(True, color=GRID, lw=0.6); a.set_axisbelow(True)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout()
fig.savefig(REPORTS_DIR / "dataset_diversity.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nn_sim, bins=30, color=C_BLUE, alpha=0.85)
ax.axvline(NEAR_DUP_THRESHOLD, color="#d03b3b", ls="--", lw=1, label=f"dedup threshold {NEAR_DUP_THRESHOLD}")
ax.set_title("Nearest-neighbour cosine similarity (lower = more diverse)")
ax.set_xlabel("cosine similarity"); ax.set_ylabel("records"); ax.legend(frameon=False)
ax.grid(True, color=GRID, lw=0.6)
fig.savefig(REPORTS_DIR / "nn_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

## 8) Stratified Split — 80 % Train / 10 % Val / 10 % Test

Split per stage so validation and test carry the same intermediate / final mix as train.
Files are written as **chat-format JSONL** (`messages` list with `system`, `user`, `assistant`
roles). The student's chat template is applied at training time by the tokenizer — section 9
renders one example to prove the format is correct.

In [ ]:
TRAIN_FRAC, VAL_FRAC = 0.80, 0.10
rng = random.Random(SEED)
splits = {"train": [], "val": [], "test": []}

for stage in sorted({r["stage"] for r in dataset}):
    bucket = [r for r in dataset if r["stage"] == stage]
    rng.shuffle(bucket)
    n = len(bucket)
    n_train, n_val = round(n * TRAIN_FRAC), round(n * VAL_FRAC)
    splits["train"] += bucket[:n_train]
    splits["val"] += bucket[n_train:n_train + n_val]
    splits["test"] += bucket[n_train + n_val:]

for name, recs in splits.items():
    rng.shuffle(recs)
    path = DATA_DIR / f"{name}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{name:<5} {len(recs):>4} records ({len(recs) / len(dataset):.0%})  "
          f"{dict(Counter(r['stage'] for r in recs))}  -> {path.relative_to(REPO_ROOT)}")

split_sizes = {k: len(v) for k, v in splits.items()}
(REPORTS_DIR / "dataset_stats.json").write_text(json.dumps(
    {"split_sizes": split_sizes, "diversity": diversity,
     "concern_counts": stats_df.concern.value_counts().to_dict(),
     "teacher_counts": stats_df.teacher.value_counts().to_dict(),
     "top_keywords": keyword_freq}, indent=2), encoding="utf-8")

## 9) Verification — Contract Audit, Leakage Check, Chat-Template Render

In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


loaded = {n: load_jsonl(DATA_DIR / f"{n}.jsonl") for n in ("train", "val", "test")}

# 1) Every saved record must still pass the quality gate.
failures = sum(
    normalize_record(r, r["teacher"], r["stage"], r["concern"]) is None
    for recs in loaded.values() for r in recs
)
print(f"Contract audit failures: {failures}")

# 2) No exact or semantic overlap between splits.
fps = {n: {record_fingerprint(r) for r in recs} for n, recs in loaded.items()}
print("Exact overlap train∩val:", len(fps["train"] & fps["val"]),
      "| train∩test:", len(fps["train"] & fps["test"]), "| val∩test:", len(fps["val"] & fps["test"]))

emb = {n: embed_texts([record_to_text(r) for r in recs]) for n, recs in loaded.items()}
for q, ref in [("val", "train"), ("test", "train"), ("test", "val")]:
    s = (emb[q] @ emb[ref].T).max(axis=1)
    print(f"{q:>4} -> {ref:<5}  max cos={s.max():.3f}  mean={s.mean():.3f}  >= {NEAR_DUP_THRESHOLD}: {(s >= NEAR_DUP_THRESHOLD).sum()}")

In [ ]:
# 3) Render one example with the STUDENT model's chat template (Qwen2.5 ChatML format).
STUDENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
student_tok = AutoTokenizer.from_pretrained(STUDENT_MODEL)
example = loaded["train"][0]
print(student_tok.apply_chat_template(example["messages"], tokenize=False))

token_lens = [len(student_tok.apply_chat_template(r["messages"], tokenize=True)) for r in loaded["train"]]
print(f"Token length (train) — p50={np.percentile(token_lens, 50):.0f}  "
      f"p95={np.percentile(token_lens, 95):.0f}  max={max(token_lens)}")

## Appendix — Full Teacher Prompt

Required by the assessment: the complete prompt sent to the teacher models. The template
lives in `task2_genai/prompts/teacher_skin_intake.md`; placeholders (`{{STAGE}}`,
`{{CONCERN}}`, …) are filled per batch by `build_prompt`. Below is the raw template and one
fully rendered example.

In [ ]:
print("=" * 30, "STUDENT SYSTEM PROMPT", "=" * 30)
print(SYSTEM_PROMPT)
print("\n" + "=" * 30, "TEACHER PROMPT TEMPLATE", "=" * 30)
print(PROMPT_TEMPLATE)
print("\n" + "=" * 30, "RENDERED EXAMPLE (final / red_flag_emergency)", "=" * 30)
print(build_prompt(SAMPLES_PER_API_CALL, "final", "red_flag_emergency"))